In [1]:
import pandas as pd
import numpy as np
import json
import pickle

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from scipy.stats import pearsonr

# M2D2 Stage 2 — Predict Drug-Drug Interactions (synergy, antagonism, additivity)

This stage of M2D2 **predicts drug–drug interaction scores** using joint profiles of **drug-target interactions** (from stage 1) as features.  
Training data (i.e. known interaction scores of drug pairs) come from literature sources.

---

## Overview
- Read in **drug-drug interaction training data**
- From stage 1 use **drug - target predictions** to create **joint profiles** for each drug pair in training data
- **Evaluate** the ability of M2D2 to predict drug combination interaction scores using **hold out** and **Pearson's r**
- **Predict** new drug–target interactions using the trained model and joint profiles

---

## Input Files Required

- **Drug-drug interaction training data**  
  → `drugInteractions_train.xlsx`  
  (This file contains known drug pairs/triplets and their interaction scores from literature.)
- **Drug-drug interactions we want the model to predict**
  → `drugInteractions_predict.xlsx` and `drugInteractions_predict2.xlsx`
  (These files contain the drug combinations we're interested in predicting)
- **Drug-target interactions from stage 1**
  → `drug_target_interactions.xlsx`
  (This file file contains single drug-target profiles from stage 1)
  
---

**Output:**
- Hold out metrics
- Predicted drug–drug interaction scores for novel combinations.


## Notes on the drug-target interactions
- we're using previously predicted drug-target interactions from stage 1 of M2D2
- this is a set of predicted drug binding affinities for all proteins in E. coli
- the drugs include all the drugs in our training set (denoted by 3 letter codes) and a set of repurposed drugs from the Broad Insitute https://repo-hub.broadinstitute.org/repurposing-app 

In [4]:
drug_target_interactions = pd.read_excel('input_data/drug_target_interactions.xlsx')

In [5]:
drug_names = drug_target_interactions.columns
drug_names

Index(['A22', 'AMK', 'AMP', 'AMX', 'AZI', 'AZT', 'BAC', 'BLM', 'BZK', 'CCCP',
       'CEC', 'CEF', 'CER', 'CFS', 'CHL', 'CIP', 'CLA', 'CSD', 'DOX', 'DXR',
       'EGCG', 'ERY', 'FOS', 'FUS', 'GEN', 'GLU', 'GLY', 'H2O2', 'ISO', 'LEV',
       'MEC', 'MEM', 'MIN', 'MXF', 'NAL', 'NIT', 'NOR', 'NVB', 'OXA', 'PAR',
       'PHL', 'PMB', 'PMS', 'PRC', 'PUR', 'PYO', 'RIF', 'SMM', 'SMX', 'SPE',
       'SPM', 'TET', 'TOB', 'TPH', 'TRC', 'TRI', 'VAN', 'VPM', 'fasudil',
       'zileuton'],
      dtype='object')

In [6]:
# get training data (drug-drug interactions from literature)
drug_interactions = pd.read_excel("input_data/drugInteractions_train.xlsx")
drug_interactions.head(5)

,drug_1,drug_2,drug_3,score,score_type,score_source,weight,weight2
0,A22,AZT,NaN,-0.178157,Bliss,E. coli BW25113 Brochado 2018 (316)E. coli IAI...,0.5,0.75
1,A22,BLM,NaN,-0.205844,Bliss,E. coli BW25113 Brochado 2018 (316)E. coli IAI...,0.5,0.75
2,A22,MEM,NaN,-0.036486,Bliss,E. coli BW25113 Brochado 2018 (316)E. coli IAI...,0.5,0.75
3,A22,PUR,NaN,0.104005,Bliss,E. coli BW25113 Brochado 2018 (316)E. coli IAI...,0.5,0.75
4,A22,PYO,NaN,0.100176,Bliss,E. coli BW25113 Brochado 2018 (316)E. coli IAI...,0.5,0.75


In [7]:
np.shape(drug_interactions)

(764, 8)

## Create the binarize function which will take the drug - target interactions and change them to 1, 0 depending on the percentile cutoff
- drug_target_interactions (continuous scores for protein-ligand interactions)
- percentile_cutoff (percentile above which a score will be considered a 1)
- drug_names (names of drugs we need binary interaction scores for, i.e. drugs that are in the training data or that we want to predict combination scores for later)

In [9]:
# set a cutoff percentile for top drug - target hits
percentile_cutoff = 80

In [10]:
def binarize(drug_target_interactions: pd.DataFrame, drug_names: list, percentile: float):
    # get only drug columns but save the protein names
    drugs_all = drug_target_interactions.columns.tolist()

    # get the top percentile hits
    p = drug_target_interactions.quantile(percentile / 100)
    binary_all = (drug_target_interactions > p).astype(int)

    # select only the drugs requested
    binary = binary_all[drug_names].to_numpy()
    
    return binary

binary = binarize(drug_target_interactions, list(drug_names), percentile_cutoff)
np.shape(binary)

(4070, 60)

## Create Join Profiles for each drug pair
Calculate a sigma (similarity) and delta (uniqueness) score for each drug pair in the training set.

We need to combine the individual drugs and their binding affinity profiles into a joint profile for each drug combination we're using for training.

The method for calculating these scores was taken from INDIGO (https://www.embopress.org/doi/full/10.15252/msb.20156777) and MAGENTA (https://journals.plos.org/ploscompbiol/article?id=10.1371/journal.pcbi.1006677).

## NOTE:
- drug array must match the order and size of the binary matrix
- rows_to_delete removes the drug-drug interactions that contain drugs not in the binary matrix (i.e. don't have available DTI data)

In [12]:
def create_sigma_delta(drug_interactions, binary, binary_drugs):
    
    # Stop if binary matrix doesn't have enough drugs
    if binary.shape[1] < len(binary_drugs):
        raise ValueError(
            f"Error: Binary matrix data does not contain enough information about all drugs.\n"
            f"We have numbers for {len(binary_drugs)} drugs but our input binary matrix has only {binary.shape[1]}"
        )
    
    # initialize the sigma/delta matrices
    num_interactions = drug_interactions.shape[0]
    num_samples = binary.shape[0]

    # sigma/delta scores are the size of drug interactions and number of proteins 
    sigma = np.zeros((num_interactions, num_samples))
    delta = np.zeros((num_interactions, num_samples))
    rows_to_delete = []

    binary_drugs_list = list(binary_drugs.values)

    for i in range(num_interactions):
        drugs = [drug_interactions.iloc[i, 0], drug_interactions.iloc[i, 1], drug_interactions.iloc[i, 2]]
        drugs_cleaned = [d for d in drugs if pd.notna(d) and str(d).lower() != 'nan']

        # if all drugs in the interaction are in binary dataset
        if all(d in binary_drugs_list for d in drugs_cleaned):
            # get scores for drugs
            logic = [binary_drugs_list.index(d) for d in drugs_cleaned]
            scores = binary[:, logic]
        
            # calculate sigma delta            
            sigma[i, :] = (np.sum(scores, axis=1) * 2 / scores.shape[1])
            delta[i, :] = (np.sum(scores > 0, axis=1) == 1).astype(int)         

        # drugs not in drug set add to rows to delete
        else:
            rows_to_delete.append(i)
            

    print(f"There are {len(rows_to_delete)} interaction scores that are not going to be considered.")
    binary_drugs_set = set(binary_drugs)
    interaction_drugs = pd.concat([
        drug_interactions['drug_1'],
        drug_interactions['drug_2'],
        drug_interactions['drug_3'].dropna()
    ])

    interaction_drugs = interaction_drugs[interaction_drugs.astype(str).str.lower() != 'nan'].unique()
    missing_drugs = [d for d in interaction_drugs if d not in binary_drugs_set]

    print(f"There is/are {len(missing_drugs)} drug(s) that is/are not in the drug names dataset:")
    print(missing_drugs)

    if not rows_to_delete:
        rows_to_delete = [0]

    return sigma, delta, np.array(rows_to_delete)

sigma, delta, r2del = create_sigma_delta(drug_interactions, binary, drug_names)

There are 0 interaction scores that are not going to be considered.
There is/are 0 drug(s) that is/are not in the drug names dataset:
[]


## Assess the M2D2 model accuracy using Hold Out and Pearson's r
- split the training data into training and testing sets
- train the model on the training set
- use the mdoel to predict the test set interactions
- compare the predicted versus true interaction scores

X --> features (sigma, delta scores)

y --> labels (drug combination interaction scores)

In [14]:
# n = 5 will give a rough performance idea but need more iterations for robustness
# n = 50 would be a better choice
n = 5
X = np.concatenate((sigma, delta), axis=1)
y = drug_interactions['score']
r_scores = []

for i in range(n):
    # split the data for hold out
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=None)

    # create and train the model for i split
    rf = RandomForestRegressor(n_estimators=100, bootstrap=True)
    rf.fit(X_train, y_train)

    # predict for the unseen test set
    y_pred = rf.predict(X_test)

    # calculate metrics
    r, p = pearsonr(y_test, y_pred)
    print(f"iteration {i+1}: r = {r:.2f}, p-value = {p:.2e}")

    # store metrics for each iteration i
    r_scores.append(r)

print(f"Mean Pearson's r over {n} iterations: {np.mean(r_scores):.2f} ± {np.std(r_scores):.2f}")

iteration 1: r = 0.68, p-value = 4.45e-33
iteration 2: r = 0.70, p-value = 3.26e-35
iteration 3: r = 0.64, p-value = 2.01e-27
iteration 4: r = 0.59, p-value = 2.10e-23
iteration 5: r = 0.74, p-value = 2.12e-41
Mean Pearson's r over 5 iterations: 0.67 ± 0.05


# PARTICIPATION ACTIVITY/QUIZ
## What if we want to use Decision Trees instead of Random Forest?
modify the code below to use Decision Trees 

Quiz Questions
- What is the mean Pearson's r over 5 iterations using Decision Trees?
- Which algorithm (random forest or decisions trees) performs better?

In [16]:
from sklearn.tree import DecisionTreeRegressor
n = 5
X = np.concatenate((sigma, delta), axis=1)
y = drug_interactions['score']
r_scores = []

for i in range(n):
    # split the data for 70/30 train/test hold out
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=None)

    # create and train the model for i split
    dt = DecisionTreeRegressor()
    dt.fit(X_train, y_train)

    # predict for the unseen test set
    y_pred = dt.predict(X_test)

    # calculate metric
    r, p = pearsonr(y_test, y_pred)
    print(f"iteration {i+1}: r = {r:.2f}, p-value = {p:.2e}")

    # store metric for each iteration i
    r_scores.append(r)

print(f"Mean Pearson's r over {n} iterations: {np.mean(r_scores):.2f} ± {np.std(r_scores):.2f}")

iteration 1: r = 0.40, p-value = 4.24e-10
iteration 2: r = 0.58, p-value = 2.36e-22
iteration 3: r = 0.49, p-value = 4.76e-15
iteration 4: r = 0.35, p-value = 5.52e-08
iteration 5: r = 0.15, p-value = 1.88e-02
Mean Pearson's r over 5 iterations: 0.39 ± 0.14


## Use M2D2 to predict interaction scores for novel combinations
- load in new combinations of interest
- create joint profiles for new combinations
- train M2D2 on all available training data
- predict interactions scores for new drug combinations

In [18]:
drug_interactions_pred = pd.read_excel("input_data/drugInteractions_predict.xlsx")
drug_interactions_pred.head()
# note that there are no interaction scores yet because M2D2 will predict them for us

,drug_1,drug_2,drug_3
0,A22,fasudil,NaN
1,AMK,fasudil,NaN
2,AMP,fasudil,NaN
3,AMX,fasudil,NaN
4,AZI,fasudil,NaN


In [19]:
sigma_pred, delta_pred, r2del_pred = create_sigma_delta(drug_interactions_pred, binary, drug_names)

There are 0 interaction scores that are not going to be considered.
There is/are 0 drug(s) that is/are not in the drug names dataset:
[]


## Train M2D2 stage 2 
## Then predict novel interactions with fasudil

fasudil is a rho kinase inhibitor used to treat vasoconstrictions from aneurysms 

In [21]:
X_train = np.concatenate((sigma, delta), axis=1)
y_train = drug_interactions['score']

# build and train model
rf = RandomForestRegressor(n_estimators=100, bootstrap=True)
rf.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [22]:
# get features and then predict labels for fasudil combinations
X_pred = np.concatenate((sigma_pred, delta_pred), axis=1)
y_pred = rf.predict(X_pred)

In [23]:
fasudil_predictions = drug_interactions_pred[['drug_1', 'drug_2']].copy()
fasudil_predictions['predicted score'] = y_pred

In [24]:
sorted_predictions = fasudil_predictions.sort_values(by='predicted score', ascending=True)
print(sorted_predictions)

   drug_1   drug_2  predicted score
13    CFS  fasudil        -0.207310
2     AMP  fasudil        -0.202775
23    FUS  fasudil        -0.197434
25    GLU  fasudil        -0.167409
53    TRI  fasudil        -0.147787
3     AMX  fasudil        -0.130487
10    CEC  fasudil        -0.129052
55    VPM  fasudil        -0.091535
54    VAN  fasudil        -0.077971
17    CSD  fasudil        -0.042426
7     BLM  fasudil        -0.033323
29    MEC  fasudil        -0.028323
41    PRC  fasudil        -0.011815
36    OXA  fasudil        -0.009161
20   EGCG  fasudil        -0.007130
0     A22  fasudil         0.021548
30    MEM  fasudil         0.024049
40    PMS  fasudil         0.027116
39    PMB  fasudil         0.027116
5     AZT  fasudil         0.028510
8     BZK  fasudil         0.039309
6     BAC  fasudil         0.051495
43    PYO  fasudil         0.064835
18    DOX  fasudil         0.076497
35    NVB  fasudil         0.094641
37    PAR  fasudil         0.097824
38    PHL  fasudil         0

# PARTICIPATION ACTIVITY/QUIZ
## What if we want to get predictions for a different repurposed drug?

Let's predict interactions for the drug zileuton, which is a drug used to treat asthma.
We already have a trained model so we just need to get predictions.

Quiz Question:
- Are there any drugs that are potentially synergistic with zileuton? If yes, which is the most synergistic?

In [26]:
drug_interactions_pred = pd.read_excel("input_data/drugInteractions_predict2.xlsx")
drug_interactions_pred.head()

,drug_1,drug_2,drug_3
0,A22,zileuton,NaN
1,AMK,zileuton,NaN
2,AMP,zileuton,NaN
3,AMX,zileuton,NaN
4,AZI,zileuton,NaN


In [27]:
sigma_pred, delta_pred, r2del_pred = create_sigma_delta(drug_interactions_pred, binary, drug_names)

There are 0 interaction scores that are not going to be considered.
There is/are 0 drug(s) that is/are not in the drug names dataset:
[]


In [28]:
X_pred = np.concatenate((sigma_pred, delta_pred), axis=1)
y_pred = rf.predict(X_pred)

In [29]:
zileuton_predictions = drug_interactions_pred[['drug_1', 'drug_2']].copy()
zileuton_predictions['predicted score'] = y_pred
sorted_predictions = zileuton_predictions.sort_values(by='predicted score', ascending=True)
print(sorted_predictions)

   drug_1    drug_2  predicted score
53    TRI  zileuton         0.262903
39    PMB  zileuton         0.311469
40    PMS  zileuton         0.311469
37    PAR  zileuton         0.323119
7     BLM  zileuton         0.328392
19    DXR  zileuton         0.386396
43    PYO  zileuton         0.395622
32    MXF  zileuton         0.464261
23    FUS  zileuton         0.467429
5     AZT  zileuton         0.485136
1     AMK  zileuton         0.489270
54    VAN  zileuton         0.505766
8     BZK  zileuton         0.512139
4     AZI  zileuton         0.514730
42    PUR  zileuton         0.518411
10    CEC  zileuton         0.523003
41    PRC  zileuton         0.531263
6     BAC  zileuton         0.538034
18    DOX  zileuton         0.538690
2     AMP  zileuton         0.542284
31    MIN  zileuton         0.560117
51    TPH  zileuton         0.562480
38    PHL  zileuton         0.582942
9    CCCP  zileuton         0.598308
3     AMX  zileuton         0.601243
17    CSD  zileuton         0.611705
1